# V3 — structured frozen NeuroLM → ZuCo sentiment

This independent notebook tests whether V1 erased useful information through global pooling. The official NeuroLM-B neural encoder remains frozen, but its output is retained as separate channel summaries and a variable-length per-second sequence. One locked attention probe is compared with a separately trained shuffled-pairing control, an inference-only structure-shuffle diagnostic, and a majority baseline.

Run V2 through its raw-cache cell first. V3 reuses those subject packs, the existing NeuroLM-B checkpoint, and no V2 performance result. Use a **GPU** Colab runtime and run every cell in order.

In [ ]:
# 1) Fetch this project, install small Colab-only NeuroLM dependencies, and pin upstream code.
from pathlib import Path
import importlib.metadata as package_metadata
import os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")
UPSTREAM_URL = "https://github.com/935963004/NeuroLM.git"
UPSTREAM_COMMIT = "0cda9876d8ce6ee07ed0c43eee5e9a6f5c24b177"
UPSTREAM_ROOT = Path("/content/NeuroLM")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "neurolm/requirements-colab.txt")])
run([sys.executable, "-c", "from huggingface_hub import is_offline_mode; import transformers; print('dependency import check passed')"])
loaded_hub = sys.modules.get("huggingface_hub")
if loaded_hub is not None and not hasattr(loaded_hub, "is_offline_mode"):
    raise RuntimeError("huggingface_hub was imported before upgrade. Restart the runtime, then rerun Cell 1.")
if not (UPSTREAM_ROOT / ".git").exists():
    UPSTREAM_ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init"], cwd=UPSTREAM_ROOT)
    run(["git", "remote", "add", "origin", UPSTREAM_URL], cwd=UPSTREAM_ROOT)
run(["git", "fetch", "--depth", "1", "origin", UPSTREAM_COMMIT], cwd=UPSTREAM_ROOT)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=UPSTREAM_ROOT)
os.chdir(PROJECT_ROOT / "neurolm")
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=Path.cwd())
print("huggingface_hub:", package_metadata.version("huggingface_hub"))
print("transformers:", package_metadata.version("transformers"))
print("Official NeuroLM commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip())

In [ ]:
# 2) Mount Drive and edit only these paths if your thesis layout differs.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/neurolm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/neurolm"
RAW_PACKS = CACHE_ROOT / "raw_eeg_packs_v2"
CHECKPOINT_ROOT = CACHE_ROOT / "upstream_checkpoints"
STRUCTURED_CACHE = CACHE_ROOT / "structured_features_v3"
RESULTS_DIR = RESULTS_ROOT / "structured_probe_v3"

if not (RAW_PACKS / "cache_manifest.json").exists():
    raise FileNotFoundError("Finish V2 Cell 3 first; its raw subject packs are V3's input")
for path in (CHECKPOINT_ROOT, STRUCTURED_CACHE, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("V2 raw packs:", RAW_PACKS)
print("V3 structured cache:", STRUCTURED_CACHE)
print("V3 results:", RESULTS_DIR)

In [ ]:
# 3) Rebuild the audited mapping, reuse/download NeuroLM-B in Drive, and initialize only its frozen encoder.
import json
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from src.channels import build_mne_spatial_mapping, select_usable_mapping
from src.config import CHECKPOINT_FILENAME, CHECKPOINT_REPOSITORY
from src.official_neurolm import OfficialNeuroLMEncoder

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")
mapping_all, mapping = select_usable_mapping(build_mne_spatial_mapping())
mapping_all.to_csv(RESULTS_DIR / "spatial_mapping.csv", index=False)
mapping_report = {
    "assignments_total": len(mapping_all),
    "assignments_used": len(mapping),
    "assignments_excluded_over_30_deg": int((~mapping_all.use_for_encoder).sum()),
    "used_mean_angular_distance_deg": float(mapping.angular_distance_deg.mean()),
    "used_max_angular_distance_deg": float(mapping.angular_distance_deg.max()),
}
(RESULTS_DIR / "spatial_mapping_diagnostics.json").write_text(json.dumps(mapping_report, indent=2))

CHECKPOINT_PATH = Path(hf_hub_download(
    repo_id=CHECKPOINT_REPOSITORY, filename=CHECKPOINT_FILENAME, local_dir=CHECKPOINT_ROOT
))
checkpoint_bytes = CHECKPOINT_PATH.stat().st_size
if checkpoint_bytes < 2_000_000_000:
    raise IOError("Checkpoint is unexpectedly small or incomplete")
encoder = OfficialNeuroLMEncoder(
    UPSTREAM_ROOT, CHECKPOINT_PATH, mapping.neurolm_index.to_numpy(),
    zuco_indices=mapping.zuco_index.to_numpy(), device="cuda"
)
provenance = {
    "checkpoint_repository": CHECKPOINT_REPOSITORY,
    "checkpoint_filename": CHECKPOINT_FILENAME,
    "checkpoint_bytes": checkpoint_bytes,
    "upstream_commit": UPSTREAM_COMMIT,
    "encoder_load": encoder.load_report,
}
(RESULTS_DIR / "checkpoint_provenance.json").write_text(json.dumps(provenance, indent=2))
print(json.dumps(mapping_report, indent=2))
print(f"Checkpoint: {CHECKPOINT_PATH} ({checkpoint_bytes / 1e9:.3f} GB)")

In [ ]:
# 4) Extract/resume one structured frozen-NeuroLM pack per subject.
from src.structured_cache import extract_structured_subject_packs

manifest = extract_structured_subject_packs(
    raw_pack_dir=RAW_PACKS,
    output_dir=STRUCTURED_CACHE,
    encoder=encoder,
    overwrite=False,
    progress_every=25,
)
print(json.dumps(manifest["report"] | {"failure_rows": len(manifest["report"]["failures"])}, indent=2))
del encoder
torch.cuda.empty_cache()

In [ ]:
# 5) Load the compact feature packs once, save diagnostics, and smoke-test the locked probe.
from src.structured_cache import load_structured_records
from src.structured_probe import StructuredProbeConfig, smoke_test_structured_probe

records, recording_metadata, dataset_report = load_structured_records(STRUCTURED_CACHE)
recording_metadata.to_csv(RESULTS_DIR / "recording_metadata.csv", index=False)
(RESULTS_DIR / "dataset_diagnostics.json").write_text(json.dumps(dataset_report, indent=2))
print(json.dumps(dataset_report, indent=2))

evaluation_config = StructuredProbeConfig(
    expected_channels=dataset_report["channels"],
    embedding_size=dataset_report["embedding_size"],
)
smoke = smoke_test_structured_probe(records, evaluation_config, device="cuda")
print(json.dumps(smoke, indent=2))

In [ ]:
# 6) Run/resume the locked 3-seed × 5-fold evaluation and save the stoplight decision.
import matplotlib.pyplot as plt
from src.structured_probe import evaluate_structured_probe

metrics, predictions, summary, delta, gate = evaluate_structured_probe(
    records=records,
    output_dir=RESULTS_DIR,
    dataset_fingerprint=dataset_report["dataset_fingerprint"],
    config=evaluation_config,
    device="cuda",
)
display(summary)
print(json.dumps(gate, indent=2))

plot_rows = metrics.groupby("setup").macro_f1.agg(["mean", "std"]).sort_values("mean")
ax = plot_rows["mean"].plot.barh(xerr=plot_rows["std"], figsize=(8, 4), capsize=3)
ax.axvline(1 / 3, color="black", linestyle="--", linewidth=1, label="balanced chance")
ax.set_xlabel("Macro-F1 across folds")
ax.set_ylabel("")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "macro_f1_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved all V3 results:", RESULTS_DIR)

## Interpretation rule

Only a **green** result may be considered for later tuning, and not until V4 is also complete. A **yellow** result is recorded without modification. A **red** result ends V3. The structure-shuffle score is diagnostic; the primary gate compares the aligned probe with the independently trained split-local shuffled-pairing control.